# Read In, Clean, and Merge Data

## Read in the data

### Read in the gated station entry data

In [37]:
# ALC
# Imports

import pandas as pd 
from os import walk
import re

In [38]:
# ALC
# specify input files. we'll start with 2022-2025 but we may narrow this down later
input_file_2022 = "../data/gated station entries/GSE_2022.csv"
input_file_2023 = "../data/gated station entries/GSE_2023.csv"
input_file_2024 = "../data/gated station entries/GSE_2024.csv"
input_file_2025 = "../data/gated station entries/GSE_2025.csv"

gse_2022 = pd.read_csv(input_file_2022)
gse_2023 = pd.read_csv(input_file_2023)
gse_2024 = pd.read_csv(input_file_2024)
gse_2025 = pd.read_csv(input_file_2025)

station_entries = pd.concat([gse_2022, gse_2023, gse_2024, gse_2025])

print(station_entries.shape)

(3731053, 7)


### Read in the service alerts data

In [39]:
# ALC
# these are as downloaded from the MBTA's Open Data Portal. There is one file
# for every month of the year. 
# todo see if we can just use the mbta api
input_filenames = [[2022, ['alerts_2022_01', 'alerts_2022_02', 'alerts_2022_03', 
                           'alerts_2022_04', 'alerts_2022_05', 'alerts_2022_06', 
                           'alerts_2022_07', 'alerts_2022_08', 'alerts_2022_09', 
                           'alerts_2022_10', 'alerts_2022_11', 'alerts_2022_12']],
                   [2023, ['alerts_2023_01', 'alerts_2023_02', 'alerts_2023_03', 
                           'alerts_2023_04', 'alerts_2023_05', 'alerts_2023_06', 
                           'alerts_2023_07', 'alerts_2023_08', 'alerts_2023_09', 
                           'alerts_2023_10', 'alerts_2023_11', 'alerts_2023_12']],
                   [2024, ['2024-01_ALERTS', '2024-02_ALERTS', '2024-03_ALERTS', 
                           '2024-04_ALERTS', '2024-05_ALERTS', '2024-06_ALERTS', 
                           '2024-07_ALERTS', '2024-08_ALERTS', '2024-09_ALERTS', 
                           '2024-10_ALERTS','2024-11_ALERTS', '2024-12_ALERTS']],
                   [2025, ['2025-01_ALERTS', '2025-02_ALERTS', '2025-03_ALERTS', 
                           '2025-04_ALERTS', '2025-05_ALERTS', '2025-06_ALERTS']]]
# initialize array
alerts = []

for year, filenames in input_filenames:
    for filename in filenames:
        path = f'../data/service alerts/{year}/{filename}.csv'
        # including low_memory = false because we have some nas in columns
        month_alerts = pd.read_csv(path, quotechar='"', low_memory = False)
        alerts.append(month_alerts)
        
# put it all together
# this is HUGE when you keep every service alert type
service_alerts = pd.concat(alerts)
print(service_alerts.shape)

(2596379, 27)


### Read in the speed restrictions data

In [40]:
#ALC

# todo see if we can just use the mbta api
slow_zone_files = []
# found this after doing service alerts above
# will focus more on switching to use the api than making consistent for now
for (root, dirs, filenames) in walk("../data/slow zones/"):
    slow_zone_files.extend(filenames)
    break
    
slow_zone_data = []

for file in slow_zone_files:
    path = f"../data/slow zones/{file}"
    if (re.search("\.csv$", file)):
        sz = pd.read_csv(path, quotechar='"')
        slow_zone_data.append(sz)
        
slow_zones = pd.concat(slow_zone_data)
slow_zones.shape

(127922, 32)

### Read in the commuter rail ridership data

In [41]:
# ALC
cr_ridership_input = "../data/commuter rail ridership/MBTA_Commuter_Rail_Ridership_by_Service_Date_and_Line.csv"
cr_ridership = pd.read_csv(cr_ridership_input)

cr_ridership.shape

(20254, 4)

### Read in the reliability data

In [42]:
# ALC
reliability_input = "../data/reliability/MBTA Bus, Commuter Rail, & Rapid Transit Reliability.csv"
reliability = pd.read_csv(reliability_input)

reliability.shape

(963838, 13)

## Clean the data and prepare for join

### Make date format consistent

In [43]:
#ALC
def get_date(date):
    return pd.to_datetime(date).dt.date

# the date in the reliability data appears yyyy/mm/dd hh:mm:ss+00
# we will use yyyy-mm-dd for service date
reliability['service_date'] = get_date(reliability['service_date'])

# cr_ridership also uses yyyy/mm/dd hh:mm:ss+00
cr_ridership['service_date'] = get_date(cr_ridership['service_date'])

# slow zones uses what we expect but we just rename the column for consistency
slow_zones = slow_zones.rename(columns = {'Calendar_Date': 'service_date'})

# service_alerts is more complicated. there appear to be some problem
# rows. for example in the june 2025 dataset there is a detour alert from july
# of 2024 that never closed
service_alerts['active_period_start_date'] = get_date(service_alerts['active_period_start_date'])
service_alerts['active_period_start_dt'] = get_date(service_alerts['active_period_start_dt'])
service_alerts['active_period_end_dt'] = get_date(service_alerts['active_period_end_dt'])
service_alerts['created_dt'] = get_date(service_alerts['created_dt'])

# gated station entries are given by the half hour. we will need to
# aggregate these down to the day
station_entries = station_entries.groupby(['service_date', 'route_or_line'], as_index=False)['gated_entries'].sum()
station_entries['service_date'] = get_date(station_entries['service_date'])

### Make line consistent

In [44]:
# ALC
# TODO make this look less HORRIFIC
def simplify_route(line):
    if pd.isna(line) or 'Silver Line' in line:
        return None
    elif 'Green Line' in line:
        return 'green'
    elif 'Red Line' in line:
        return "red"
    elif 'Blue Line' in line:
        return 'blue'
    elif 'Orange Line' in line:
        return 'orange'
    elif 'Framingham/Worcester' in line or line == 'Worcester Line Shuttle':
        return 'framingham-worcester'
    elif 'Haverhill' in line:
        return 'haverhill'
    elif 'Kingston' in line:
        return 'kingston-plymouth'
    elif 'Lowell' in line:
        return 'lowell'
    elif 'Needham' in line:
        return 'needham'
    elif 'Fitchburg' in line:
        return 'fitchburg'
    elif 'Greenbush' in line:
        return 'greenbush'
    elif 'Fairmount' in line:
        return 'fairmount'
    elif 'Newburyport/Rockport' in line:
        return 'newburyport-rockport'
    elif 'Providence/Stoughton' in line:
        return 'providence-stoughton'
    elif 'Middleborough/Lakeville' in line:
        return 'middleborough-lakeville'
    elif 'Foxboro' in line or 'Franklin' in line:
        return 'franklin-foxboro'
    elif line == 'Fall.River/New.Bedford':
        return 'fallriver-newbedford'
    else:
        return line

# get cr_ridership lines consistent with other datasets
cr_ridership['line'] = cr_ridership['line'].apply(simplify_route)
cr_ridership = cr_ridership.rename(columns = {'estimated_boardings': 'est_ridership'})

# station entry lines
station_entries['route_or_line'] = station_entries['route_or_line'].apply(simplify_route)

# there is silver line data in here which is out of scope
station_entries = station_entries[station_entries['route_or_line'].isin(['green', 'red', 'blue', 'orange'])]
station_entries = station_entries.rename(columns = {'gated_entries': 'est_ridership',
                                                   'route_or_line': 'line'})

# slow zones lines
slow_zones['line'] = slow_zones['Line'].apply(simplify_route)

# reliability lines, cr and rt
reliability['line'] = reliability['gtfs_route_long_name'].apply(simplify_route)

# this removes out of scope bus data
reliability = reliability[reliability['line'].notna()]

# get one row per line per date with an aggregate score
reliability = reliability.groupby(['service_date', 'line'], as_index=False)[['otp_numerator', 'otp_denominator']].sum()

## Merge the data

### We need to get our ridership data compiled together

In [45]:
ridership = pd.concat([station_entries, cr_ridership], ignore_index = True)

In [53]:
reliability_cr = pd.merge(cr_ridership, reliability, on=['service_date', 'line'], how='inner')
reliability_cr['otp_score'] = reliability_cr['otp_numerator'] / reliability_cr['otp_denominator']
reliability_cr[reliability_cr['line'] == 'franklin-foxboro'].corr(numeric_only = True)

,est_ridership,ObjectId,otp_numerator,otp_denominator,otp_score
est_ridership,1.000000,0.620070,0.045814,0.091466,-0.154502
ObjectId,0.620070,1.000000,-0.482430,-0.482405,-0.069857
otp_numerator,0.045814,-0.482430,1.000000,0.947629,0.225589
otp_denominator,0.091466,-0.482405,0.947629,1.000000,-0.078411
otp_score,-0.154502,-0.069857,0.225589,-0.078411,1.000000
